# 03 · Selección de variables (solo 2023–2024)

Primera fase del protocolo (descrito en la metodología del documento):

- **Datos:** solo desembolsos de 2023 y 2024. **2025 está sellado**: `protocolo.cargar_desarrollo()` no lo devuelve y lo verifica.
- **Objetivo:** mora 30+ a 6 meses; los pagos anticipados (salidas del panel sin deterioro) cuentan como buenos pagadores.
- **Métrica:** AUC **dentro de cada cosecha** (2023 y 2024 por separado, promedio ponderado), en validación cruzada 5×5 estratificada por año × evento.
- **Variables candidatas:** las 17 permitidas (notebook 02, sección 5). El sexo y el código de distribuidor se muestran solo como referencia.

Scripts: `src/03a_revision_variables.py` (revisión univariada) y `src/03b_seleccion_variables.py` (eliminación hacia atrás).

In [1]:
import json, sys, importlib.util, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
import protocolo as P
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 200); pd.set_option("display.float_format", "{:,.4f}".format)
R, R25, DG = P.SALIDA, P.SALIDA / "reporte_2025", P.SALIDA / "diagnosticos"
leer = lambda r: pd.read_csv(r, encoding="utf-8-sig")
def script(nombre):
    spec = importlib.util.spec_from_file_location(nombre.replace(".py", ""), RAIZ / "src" / nombre)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

## 1. 2025 está sellado

In [2]:
dev = P.cargar_desarrollo()
print("Años:", sorted(int(a) for a in dev["anio"].unique()), "| créditos maduros:", len(dev), "| con resultado a 6 meses:", dev[P.Y].notna().sum())
print("Tasa de mora 30+ a 6 meses por año:", dev.groupby("anio")[P.Y].mean().round(3).to_dict())

Años: [2023, 2024] | créditos maduros: 1875 | con resultado a 6 meses: 1875
Tasa de mora 30+ a 6 meses por año: {2023: 0.348, 2024: 0.186}


## 2. Revisión univariada

Para cada variable: nulos, información (IV), AUC univariado con la variable codificada por la tasa del evento **dentro del pliegue**, AUC dentro de 2023 y de 2024 por separado, y la V de Cramér con el año (cuánto distingue la cosecha). Es **descriptiva**: no decide qué variables quedan.

In [3]:
revision = script("03a_revision_variables.py")
revision.main()

                variable                     grupo  pct_nulos  n_categorias  pct_valor_dominante     iv  p_chi2  auc_cv_intra_cosecha  auc_cv_de  v_cramer_anio  auc_cv_2023  auc_cv_2024
     codigo_distribuidor excluida (solo auditoría)     0.0000            97               0.1670 0.4520  0.0000                0.5840     0.0090         0.3490       0.5900       0.5910
                    sexo excluida (solo auditoría)     0.0000             2               0.6510 0.0270  0.0030                0.5310     0.0030         0.0000       0.5010       0.5410
         zona_homologada                     lugar     0.0210             6               0.3690 0.1140  0.0000                0.5640     0.0020         0.2610       0.5420       0.5490
            departamento                     lugar     0.0070            25               0.3870 0.1820  0.0000                0.5600     0.0050         0.2800       0.5440       0.5870
         log_frec_ciudad                     lugar     0.0000         

**Lectura:**
- Entre las permitidas, las de mayor señal son la **familia de moto**, el **tipo de uso / cilindraje / precio**, la **geografía** (zona, ciudad, departamento, frecuencia de la ciudad) y el **estado civil**, con AUC univariados entre 0,553 (frecuencia de la ciudad) y 0,597 (familia de moto).
- La **edad** no ordena por sí sola en el agrupado (0,484), pero sí dentro de cada año (0,526 en 2023 y 0,542 en 2024): su relación con la mora no es lineal y cambia entre cosechas.
- El **código de distribuidor**, excluido del modelo, tiene una señal univariada de 0,584, pero varía mucho con el año (V = 0,35): es en buena parte la mora del canal en cada periodo. El **sexo** aporta poco por sí solo (0,531).
- Hay **redundancia** fuerte en dos grupos: geografía (ciudad, zona, departamento, frecuencia de la ciudad; V ≥ 0,86) y producto (familia, tipo de uso, cilindraje, precio; V ≥ 0,72). La selección multivariada decide cuáles sobran.

## 3. Selección por eliminación hacia atrás

**Regla (registrada antes de ejecutar los modelos):** se parte de las 17 permitidas y, con un LightGBM de parámetros fijos como modelo de tamizaje, se calcula el **aporte** de cada variable (AUC dentro de cosecha con todas menos el AUC sin ella, promedio de 25 particiones). Se quita la de menor aporte mientras ese aporte sea **≤ 0**, es decir, mientras quitarla no baje el AUC promedio.

In [4]:
cam = leer(R / "seleccion_camino.csv")
sel = json.loads((R / "seleccion_variables.json").read_text(encoding="utf-8"))
elim = cam[cam["eliminada"].astype("boolean").fillna(False).astype(bool)][["paso", "variable"]]
aportes = cam[cam["aporte"].notna()]
camino = elim.merge(aportes[["paso", "variable", "aporte", "auc_con_todas", "pliegues_aporte_positivo"]], on=["paso", "variable"])
camino.index = range(1, len(camino) + 1)
camino["pliegues_aporte_positivo"] = camino["pliegues_aporte_positivo"].astype(int)
display(camino.rename(columns={"variable": "variable eliminada", "auc_con_todas": "AUC antes de quitarla",
                               "pliegues_aporte_positivo": "pliegues con aporte > 0 (de 25)"}))
ultimo = aportes[aportes["paso"] == aportes["paso"].max()].sort_values("aporte")
display(ultimo[["variable", "aporte", "ic_inf", "ic_sup", "pliegues_aporte_positivo"]].astype({"pliegues_aporte_positivo": int}).rename(columns={"variable": "variable que queda"}).reset_index(drop=True))
print("Seleccionadas (C2):", sel["variables"])
print(f"AUC dentro de cosecha del tamizaje: {cam['auc_con_todas'].dropna().iloc[0]:.4f} con las 17 -> {sel['auc_intra_cosecha']:.4f} con las {len(sel['variables'])}")

,paso,variable eliminada,aporte,AUC antes de quitarla,pliegues con aporte > 0 (de 25)
1,0,vol_distribuidor_rango,-0.0050,0.6335,6
2,1,nivel_estudio,-0.0022,0.6385,11
3,2,ciudad,-0.0037,0.6406,8
4,3,personas_a_cargo,-0.0034,0.6444,9
5,4,independiente_informal,-0.0018,0.6478,12


,variable que queda,aporte,ic_inf,ic_sup,pliegues_aporte_positivo
0,precio_moto,0.0013,-0.0057,0.0083,15
1,departamento,0.0018,-0.0135,0.0172,14
2,propietario_moto_cliente,0.0022,-0.0047,0.0092,14
3,log_frec_ciudad,0.0027,-0.0107,0.0160,13
4,tipo_uso_moto,0.0033,-0.0025,0.0092,18
5,compra_fuera_depto,0.0038,-0.0042,0.0117,18
6,cilindraje,0.0041,-0.0015,0.0097,21
7,zona_homologada,0.0045,-0.0039,0.0129,17
8,estado_civil,0.0060,-0.0108,0.0227,15
9,familia_moto,0.0073,-0.0056,0.0202,18


Seleccionadas (C2): ['edad', 'estado_civil', 'estrato', 'zona_homologada', 'departamento', 'compra_fuera_depto', 'log_frec_ciudad', 'familia_moto', 'tipo_uso_moto', 'cilindraje', 'precio_moto', 'propietario_moto_cliente']
AUC dentro de cosecha del tamizaje: 0.6335 con las 17 -> 0.6496 con las 12


**Lectura:**
- Salen, en orden, el **volumen del distribuidor, el nivel de estudio, la ciudad, las personas a cargo y el tipo de independiente**: quitar cada una no bajaba el AUC, y al quitarlas el AUC del tamizaje sube de 0,633 a 0,650.
- Quedan **12 variables** (versión **C2**): edad, estado civil, estrato, zona, departamento, compra fuera del departamento, frecuencia de la ciudad, familia de moto, tipo de uso, cilindraje, precio y moto a nombre del cliente.
- La regla es deliberadamente conservadora: conserva variables redundantes entre sí (geografía, producto) si cada una aporta algo positivo. Por eso C2 compite en la fase siguiente con versiones más simples, y la regla de decisión del protocolo elige la más simple que no se distinga de la mejor (notebook 04).

## 4. Versiones candidatas que pasan a la siguiente fase

In [5]:
pd.DataFrame([{"versión": v, "n.º de variables": len(vs), "variables": ", ".join(vs)} for v, vs in P.candidatas().items()])

,versión,n.º de variables,variables
0,C1_completo,17,"edad, estado_civil, estrato, personas_a_cargo, nivel_estudio, independiente_informal, ciudad, zona_homologada, departamento, compra_fuera_depto, log_frec_ciudad, vol_distribuidor_rango, familia_mo..."
1,C3_V2p,6,"edad, estado_civil, ciudad, independiente_informal, nivel_estudio, familia_moto"
2,C4_V2p_compra,7,"edad, estado_civil, ciudad, independiente_informal, nivel_estudio, familia_moto, compra_fuera_depto"
3,C5_V3p,9,"edad, estado_civil, ciudad, independiente_informal, nivel_estudio, familia_moto, compra_fuera_depto, log_frec_ciudad, vol_distribuidor_rango"
4,C6_V5p,10,"edad, estado_civil, ciudad, independiente_informal, nivel_estudio, familia_moto, compra_fuera_depto, log_frec_ciudad, vol_distribuidor_rango, departamento"
5,C2_seleccionado,12,"edad, estado_civil, estrato, zona_homologada, departamento, compra_fuera_depto, log_frec_ciudad, familia_moto, tipo_uso_moto, cilindraje, precio_moto, propietario_moto_cliente"


Las versiones C3 a C6 parten de una selección inicial de variables de la persona y del producto, anterior al protocolo (incluyó los desembolsos de 2025, por lo que este protocolo la reemplaza): C3 tiene seis variables; C4 le agrega la compra fuera del departamento; C5, la frecuencia de la ciudad y el volumen del distribuidor; y C6, el departamento. C1 son las 17 permitidas.